In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, accuracy_score

In [ ]:
# pip install tensorflow  # Only if TensorFlow is not installed

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

In [ ]:
# Locate the CSV when running from S4 or the repository root
paths = [
    Path('Churn_Modelling.csv'),
    Path('../SV/ML/2_Bank_Customer_Churn/Churn_Modelling.csv'),
    Path('LP3/SV/ML/2_Bank_Customer_Churn/Churn_Modelling.csv')
]
csv_path = next((path for path in paths if path.is_file()), None)
if csv_path is None:
    raise FileNotFoundError('Churn_Modelling.csv not found. Place it next to this notebook.')
dataset = pd.read_csv(csv_path, index_col='RowNumber')

In [ ]:
dataset.head()

In [ ]:
# Features and target
X = dataset.iloc[:, 2:12]
X  # drop CustomerId and Surname

In [ ]:
# Features and target
Y = dataset.iloc[:, 12].values  # Churn column
Y

We are treating countries with ordinal values(0 < 1 < 2) but they are incomparable.
To solve this we can use one hot encoding.
We will perform some standardization

In [ ]:
# Pipeline: One-hot encode categorical and scale
pipeline = Pipeline([
    ('preprocess', ColumnTransformer(
        transformers=[
            ('gender', OneHotEncoder(drop='first'), ['Gender']),
            ('geo', OneHotEncoder(drop='first'), ['Geography'])
        ],
        remainder='passthrough'
    )),
    ('scaler', StandardScaler())
])

In [ ]:
# Split first to avoid data leakage
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=0)

In [ ]:
# Standardize features using training data only
X_train = pipeline.fit_transform(X_train)
X_test = pipeline.transform(X_test)

In [ ]:
# Build ANN
classifier = Sequential()
classifier.add(Dense(6, activation='relu', input_shape=(X_train.shape[1],)))
classifier.add(Dropout(0.1))
classifier.add(Dense(6, activation='relu'))
classifier.add(Dropout(0.1))
classifier.add(Dense(1, activation='sigmoid'))

In [ ]:
# Compile and train ANN
classifier.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
history = classifier.fit(X_train, y_train, batch_size=32, epochs=100, validation_split=0.1, verbose=2)

In [ ]:
# Predict
y_pred = (classifier.predict(X_test) > 0.5).astype(int)

In [ ]:
y_pred = classifier.predict(X_test)
print(y_pred[:5])

In [ ]:
# Let us use confusion matrix with cutoff value as 0.5
y_pred = (y_pred > 0.5).astype(int)
print(y_pred[:5])

In [ ]:
# Making the Matrix
cm = confusion_matrix(y_test, y_pred)
print(cm)

In [ ]:
# Accuracy of our NN
print(((cm[0][0] + cm[1][1]) * 100) / len(y_test), '% of data was classified correctly')